# NorthRiver Data Warehouse — Data Dictionary

## fact_orders
**Grain:** One row per order line item.  
**Populated by:** `05_load` (following extraction via `03_extract` and transformation via `04_transform`)

| Column | Data Type | Description |
| :--- | :--- | :--- |
| order_line_key | BIGINT | Primary key; unique surrogate key for the order line. |
| order_id | STRING | Natural business identifier of the parent order. |
| order_line_number | INT | Position of the line within the order. |
| order_date_key | INT | Foreign key to dim_date (YYYYMMDD format). |
| customer_key | BIGINT | Foreign key to dim_customer; resolves to the customer version valid on the order date. |
| product_key | BIGINT | Foreign key to dim_product; resolves to the product version valid on the order date. |
| channel_key | INT | Foreign key to dim_channel. |
| quantity | INT | Number of units sold on this line. |
| unit_price | DECIMAL(10,2) | Price per unit at time of sale. |
| discount_amount | DECIMAL(10,2) | Discount applied to the line. |
| net_amount | DECIMAL(12,2) | Line revenue net of discounts. |
| load_ts | TIMESTAMP | Pipeline processing time when the row was loaded. |

---

## dim_customer (SCD Type 2)
**Grain:** One row per customer per version.  
**Populated by:** `04_transform` and `05_load`

| Column | Data Type | Description |
| :--- | :--- | :--- |
| customer_key | BIGINT | Primary key; unique surrogate key assigned to each customer version. |
| customer_id | STRING | Natural business identifier of the customer; repeats across versions. |
| customer_name | STRING | Full name of the customer. |
| email | STRING | Customer contact email address. |
| segment | STRING | Customer segment (e.g., Retail, Wholesale, Enterprise). |
| region | STRING | Geographic region of the customer. |
| signup_date | DATE | Date the customer account was created. |
| valid_from | TIMESTAMP | Effective start of this version's validity. |
| valid_to | TIMESTAMP | Effective end of validity; NULL for the current version. |
| is_current | BOOLEAN | TRUE for the active version of each customer. |
| hash_diff | STRING | Hash of tracked attributes used to detect changes for SCD2 processing. |
| load_ts | TIMESTAMP | Pipeline processing time when the row was loaded. |

---

## dim_product (SCD Type 2)
**Grain:** One row per product per version.  
**Populated by:** `04_transform` and `05_load`

| Column | Data Type | Description |
| :--- | :--- | :--- |
| product_key | BIGINT | Primary key; unique surrogate key assigned to each product version. |
| product_id | STRING | Natural business identifier of the product; repeats across versions. |
| product_name | STRING | Display name of the product. |
| category | STRING | Product category (e.g., Apparel, Electronics, Home). |
| subcategory | STRING | Product subcategory within the category. |
| brand | STRING | Brand or manufacturer of the product. |
| unit_cost | DECIMAL(10,2) | Standard cost per unit for this version. |
| list_price | DECIMAL(10,2) | List price per unit for this version. |
| status | STRING | Product lifecycle status. |
| valid_from | TIMESTAMP | Effective start of this version's validity. |
| valid_to | TIMESTAMP | Effective end of validity; NULL for the current version. |
| is_current | BOOLEAN | TRUE for the active version of each product. |
| hash_diff | STRING | Hash of tracked attributes used to detect changes for SCD2 processing. |
| load_ts | TIMESTAMP | Pipeline processing time when the row was loaded. |

---

## dim_channel
**Grain:** One row per sales channel (stable lookup, no SCD history tracking).  
**Populated by:** `04_transform` and `05_load`

| Column | Data Type | Description |
| :--- | :--- | :--- |
| channel_key | INT | Primary key; stable surrogate key for the channel. |
| channel_code | STRING | Natural business identifier of the channel (e.g., WEB, STORE). |
| channel_name | STRING | Human-readable name of the channel. |

---

## dim_date
**Grain:** One row per calendar day (fixed lookup table, no SCD).  
**Populated by:** Initial schema setup scripts.

| Column | Data Type | Description |
| :--- | :--- | :--- |
| date_key | INT | Primary key; date in YYYYMMDD integer format. |
| full_date | DATE | Actual calendar date. |
| year | INT | Four-digit calendar year. |
| month | INT | Calendar month number (1–12). |

# NorthRiver Pipeline Lineage

Data flows from the operational source database through staging into the analytical data warehouse.

## Pipeline Flow Overview

`PostgreSQL OLTP Database` → `northriver_staging` → `northriver_warehouse`

### Step 1: Extract
* **Source Tables:** `stg_orders`, `stg_customers`, `stg_products`, `stg_channels` from PostgreSQL.
* **Schema:** `northriver_staging`
* **Handled by:** Saved SQL query `03_extract`
* **Transformation:** Ingests raw operational data into staging tables.

### Step 2: Transform
* **Schema:** `northriver_staging` / intermediate models
* **Handled by:** Saved SQL query `04_transform`
* **Transformation:** Cleans staging data and prepares transformations, including hash generation and attribute tracking for SCD Type 2 dimensions.

### Step 3: Load
* **Warehouse Schema:** `northriver_warehouse` (`fact_orders`, `dim_customer`, `dim_product`, `dim_channel`, `dim_date`)
* **Handled by:** Saved SQL query `05_load`
* **Transformation:** Implements SCD Type 2 logic for dimensions (surrogate key generation, effective date bounding) and loads transactional data into `fact_orders` using point-in-time surrogate key joins.
